In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:


# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
X_batch, y_batch = next(iter(train_loader))
print(f"Training batch input shape: {X_batch.shape}")
print(f"Training batch labels shape: {y_batch.shape}")

In [ ]:
# Get one batch of images and labels
images, labels = next(iter(train_loader))

# Display the first 6 images in the batch
plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)

    # Convert from (C, H, W) to (H, W, C) for matplotlib
    img = images[i].permute(1, 2, 0)

    plt.imshow(img)
    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
class NN4Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super(NN4Layer, self).__init__()

        # TODO: What are the dimensions of the first layer? (hint: features? hidden neurons?)
        self.layer1 = nn.Linear(input_dim, hidden_dim)

        # TODO: What are the dimensions from previous and next layer?
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)

        # TODO: What are the dimensions from previous and next layer?
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)

        # TODO: What are the dimensions from previous and next layer?
        self.layer4 = nn.Linear(hidden_dim, output_dim)

        # activation function for non-linearity
        self.relu = nn.ReLU()

    # Defines how input data flows through the network
    def forward(self, x):
        # Layer 1: TODO
        z1 = self.layer1(x)
        a1 = self.relu(z1)

        # Layer 2 TODO
        z2 = self.layer2(a1)
        a2 = self.relu(z2)

        # Layer 3 TODO
        z3 = self.layer3(a2)
        a3 = self.relu(z3)

        # Output layer TODO
        z4 = self.layer4(a3)

        # TODO: are we missing output activation function? (hint: Loss Function)

        return z4

In [ ]:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
    # Set the model to training mode
    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        # Move batch to the selected device
        X_batch = X_batch.view(X_batch.size(0), -1).to(device)
        y_batch = y_batch.to(device)

        # Forward pass
        # TODO: make predictions
        outputs = model(X_batch) # shape: (batch_size, 10)
        # TODO: compute loss
        loss = criterion(outputs, y_batch)

        # Backward pass & optimization
        optimizer.zero_grad()   # Clear previous gradients
        loss.backward()         # Compute gradients
        optimizer.step()        # Update model parameters

        running_loss += loss.item()

    # Average loss over all batches
    avg_loss = running_loss / len(train_loader)

    return avg_loss

In [ ]:
def validate(model, criterion, test_loader, device):
  # Set the model to evaluation mode
  model.eval()

  running_loss = 0.0

  with torch.no_grad():
    for X_batch, y_batch in test_loader:
      # Move data to device
      X_batch = X_batch.view(X_batch.size(0), -1).to(device)
      y_batch = y_batch.to(device)


      # Forward pass (continuous output)
      outputs = model(X_batch)                   # shape: (batch_size, 1)
      loss = criterion(outputs, y_batch)

      running_loss += loss.item()

  # Average loss over all batches
  avg_loss = running_loss / len(test_loader)

  return avg_loss

In [ ]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model parameters
# TODO: What are input features ? (hint: flatten using channel, height, width)
input_dim = 3 * 36 * 36

# TODO: choose number of hidden neurons
hidden_dim = 32          # Design choice

# TODO: what are the number of classes ?
output_dim = 1
# TODO: Instantiate model (what are the model class inputs?)
model = NN4Layer(input_dim, hidden_dim, output_dim).to(device)

# TODO: Print the model architecture
print("Model Architecture:\n")
print(model)

# Calculate the total number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")


In [ ]:
# TODO: choose number of epochs, increase it if you have gpus :)
num_epochs = 20
# TODO: choose a learning rate (try different values and evaluate results)
learning_rate = 0.001

# Define criterion (loss function)
criterion = nn.MSELoss()
# Define optimizer
optimizer = AdamW(model.parameters(), learning_rate)

In [ ]:
# Run Training
train_losses = []
val_losses = []

print('Starting Training...')
for epoch in range(num_epochs):
  # Train one epoch
  train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

  # Validate
  val_loss = validate(model, criterion, test_loader, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  if (epoch + 1) % 5 == 0:
    print(
      f'Epoch [{epoch+1}/{num_epochs}], '
      f'Train Loss: {train_loss:.4f}, '
      f'Val Loss: {val_loss:.4f}'
    )

print('Training Complete!')

In [ ]:
# Plotting results
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Set model to evaluation mode
model.eval()
# Get one batch from the test DataLoader
images, labels = next(iter(test_loader))
# Move images to device
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    # Flatten images before passing to the model
    outputs = model(images.view(images.size(0), -1))
    predictions = outputs

# Move tensors back to CPU for plotting
images = images.cpu()
labels = labels.cpu()
predictions = predictions.cpu()

# Get one batch of images and labels
images, labels = next(iter(train_loader))

# Display the first 6 images in the batch
plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)

    # Convert from (C, H, W) to (H, W, C) for matplotlib
    img = images[i].permute(1, 2, 0)

    plt.imshow(img)
    plt.title(f"True: {labels[i]} | Pred: {predictions[i][0]}")
    plt.axis('off')

plt.tight_layout()
plt.show()